# 1.IMPORT AND BASIC UNDERSTANDING OF THE DATA

 1.MOUNTING DATA FROM DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# unzip the folder and save to the created directory
!mkdir -p /content/dataset
!unzip -qo /content/drive/MyDrive/Vehicle_Project_Data/dataset.zip -d /content/dataset

print("Dataset succsessfully unzipped!!")

Dataset succsessfully unzipped!!


 2.UNDERSTAND THE INSTANCES

In [ ]:
import glob
import os
import pandas as pd

# search any csv files in the folder or sub folder
csv_files=glob.glob('dataset/**/*.csv',recursive=True)

if not csv_files:
    print("Error: No CSV files found anywhere inside the 'export' folder.")
    print("\nHere is what we actually found in 'export':")
    # Print the directory tree so you can see what's going on
    for root, dirs, files in os.walk('export'):
        for file in files:
            print(os.path.join(root, file))
else:
    #use the founded csv file
    target_csv=csv_files[0]
    print(f"Succsess! find the csv file: {target_csv}\n")

    try:
        df=pd.read_csv(target_csv)

        print(f"Total Rows & Columns: {df.shape}")
        print(f"Columns: {list(df.columns)}")

        print("\nFirst 5 Rows:")
        print(df.head())

        # Check for the class column dynamically to avoid KeyError if named differently
        class_col='class' if 'class' in df.columns else (df.columns[-1] if len(df.columns) > 0 else None)

        if class_col and class_col in df.columns:
            print(f"\nDistribution for column '{class_col}':")
            print(df[class_col].value_counts())
        else:
            print("\nCould not determine a class/label column.")

    except Exception as e:
        print(f"An error occurred while reading the CSV: {e}")

Succsess! find the csv file: dataset/export/_annotations.csv

Total Rows & Columns: (194539, 8)
Columns: ['filename', 'width', 'height', 'class', 'xmin', 'ymin', 'xmax', 'ymax']

First 5 Rows:
                                            filename  width  height  \
0  1478900859981702684_jpg.rf.6830635c7d919747563...    512     512   
1  1478900859981702684_jpg.rf.6830635c7d919747563...    512     512   
2  1478900859981702684_jpg.rf.6830635c7d919747563...    512     512   
3  1478900859981702684_jpg.rf.6830635c7d919747563...    512     512   
4  1478900859981702684_jpg.rf.6830635c7d919747563...    512     512   

        class  xmin  ymin  xmax  ymax  
0         car   291   247   370   331  
1  pedestrian   270   235   293   321  
2         car     0   266    13   327  
3         car    25   258   106   304  
4         car   111   259   135   289  

Distribution for column 'class':
class
car                        127873
pedestrian                  21491
trafficLight-Red            1367

## 2.FEATURE EXCTRACTION AND MODEL BUILDING


 2.1.IMPORT LIBRARIES

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report
import joblib

# Configurations
IMAGE_DIR='dataset/export' # Pointing to unzipped target folder
IMG_SIZE=(64,64)

  2.2.DEFINE MODULAR PREPROCESSING FUNCTIONS

In [ ]:
def preprocess_for_model(cropped_img,model_type='svm'):
    if cropped_img is None or cropped_img.size == 0:
        return None

    resized_img=cv2.resize(cropped_img, (64, 64))

    if model_type=='svm':
        # Grayscale ஆக மாற்றவும்
        if len(resized_img.shape)==3:
            gray=cv2.cvtColor(resized_img,cv2.COLOR_BGR2GRAY)
        else:
            gray=resized_img

        # HOG Feature Extraction
        hog_features=hog(
            gray,
            orientations=9,
            pixels_per_cell=(8,8),
            cells_per_block=(2,2),
            visualize=False
        )
        return hog_features

    else:
        features=resized_img.flatten()
        return features

 3.GENERALIZED DATA GENERATOR

In [ ]:
def load_dataset_pipeline(df,class_targets,model_type='svm'):
    """
    Loops through the DataFrame to crop targets and extract features
    until specific per-class target limits are met.

    class_targets: dict mapping class names to maximum desired counts.
    """
    x=[]
    y=[]

    # Initialize counts for each tracked class
    counts={cls: 0 for cls in class_targets.keys()}

    # Shuffle to ensure we mix up the data as we sample
    sampled_df=df.sample(frac=1, random_state=42)

    for index,row in sampled_df.iterrows():
        label=row['class']

        #Skip if it's a class we don't care about, or if we already have enough of it
        if label not in class_targets or counts[label]>=class_targets[label]:
            # Optional check: If every single target is satisfied, break out early to save time
            if all(counts[c]>=class_targets[c] for c in class_targets):
                break
            continue

        img_path=os.path.join(IMAGE_DIR, row['filename'])
        img=cv2.imread(img_path)
        if img is None:
            continue

        xmin,ymin,xmax,ymax=row['xmin'],row['ymin'],row['xmax'],row['ymax']
        cropped=img[ymin:ymax,xmin:xmax]

        if cropped.size==0:
            continue

        # Call the modular preprocessing pipeline
        features=preprocess_for_model(cropped,model_type=model_type)

        x.append(features)
        y.append(label)

        #Increment the specific class counter
        counts[label]+=1

    print(f"Extraction complete! Collected counts: {counts}")
    return np.array(x,dtype=np.float32),np.array(y)


# Define your custom required target numbers
# Note: Ensure these strings exactly match the names in your df['class'] column
my_targets={
    'car': 40000,
    'biker': 3704,
    'truck': 7194
}

# Load data specifically prepared for SVM
print("Extracting features from dataset...")
x,y=load_dataset_pipeline(df,class_targets=my_targets, model_type='svm')
print(f"Features extracted successfully! Shape: {x.shape}")

Extracting features from dataset...
Extraction complete! Collected counts: {'car': 40000, 'biker': 3704, 'truck': 7194}
Features extracted successfully! Shape: (50898, 1764)


 4.TRAIN, EVALUATE, AND EXPORT THE MODEL

*Model Training

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

*  Define and train SVM Classifier
* Feature Scaling

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV

# Feature Scaling for Quick Training
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

# Quickest LinearSVC
base_svm=LinearSVC(dual=False,C=1.0,random_state=42,max_iter=1000)

# to measure backround probability
svm_classifier=CalibratedClassifierCV(base_svm,cv=3)

print("Training optimized Linear SVM...")
svm_classifier.fit(X_train_scaled,y_train)
print("Training complete!")

Training optimized Linear SVM...
Training complete!


* Model Evaluation

In [ ]:
y_pred=svm_classifier.predict(X_test_scaled)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

       biker       0.88      0.68      0.77       741
         car       0.92      0.97      0.95      8000
       truck       0.85      0.71      0.77      1439

    accuracy                           0.91     10180
   macro avg       0.89      0.79      0.83     10180
weighted avg       0.91      0.91      0.91     10180



* Save trained SVM model for the speed estimation script

In [ ]:
model_path='vehicle_svm_detector.joblib'
joblib.dump(svm_classifier,model_path)
print(f"Model saved successfully to '{model_path}'!")

Model saved successfully to 'vehicle_svm_detector.joblib'!
